# 4 - TRANSFORMATION

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

source= Path(r"filepath\csv_clean")

stocks= pd.read_csv(source / "stocks_clean.csv", parse_dates=["date"],date_format="%Y-%m-%d")

index= pd.read_csv(source / "index_clean.csv", parse_dates=["date"], date_format="%Y-%m-%d")

companies= pd.read_csv(source / "companies_clean.csv") #there's no date column in this table.


Ordering the df:

In [ ]:
#Make sure the tables are sorted by symbol and date

stocks= stocks.sort_values(['symbol','date']).reset_index(drop=True)
index= index.sort_values(["date"]).reset_index(drop=True)


### STOCKS table

Simples return and Log return:

In [ ]:
grouped=stocks.groupby("symbol")["adj_close"]

stocks["simple_return"]=grouped.pct_change()

stocks["log_return"]=np.log(stocks["adj_close"]/grouped.shift(1))

'\norder columns of a df:\n\n   col = df.pop("log_return")\n   df.insert(3, "log_return", col)\n\n   OR\n\n   cols = [\n    "date",\n    "symbol",\n    "close",\n    "log_return",\n    "high",\n    "low",\n    "open",\n    "volume"\n    ]\n    #in the order i want\n\n    df = df[cols]\n'

Cumulative return since the first ever price of each symbol:

In [ ]:
first_price= grouped.transform("first")

stocks["cumulative_return"]=(stocks["adj_close"]/first_price)-1

stocks

,date,symbol,adj_close,close,high,low,open,volume,no_trading,simple_return,log_return,cumulative_return
0,2010-01-04,A,20.053030,22.389128,22.625179,22.267525,22.453505,3815561.0,False,NaN,NaN,0.000000
1,2010-01-05,A,19.835201,22.145924,22.331903,22.002861,22.324751,4186031.0,False,-0.010863,-0.010922,-0.010863
2,2010-01-06,A,19.764729,22.067240,22.174536,22.002861,22.067240,3243779.0,False,-0.003553,-0.003559,-0.014377
3,2010-01-07,A,19.739098,22.038628,22.045780,21.816881,22.017166,3095172.0,False,-0.001297,-0.001298,-0.015655
4,2010-01-08,A,19.732698,22.031473,22.067240,21.745352,21.917025,3733918.0,False,-0.000324,-0.000324,-0.015974
...,...,...,...,...,...,...,...,...,...,...,...,...
1791056,2024-12-16,ZTS,175.809998,175.809998,179.289993,175.789993,177.990005,1938800.0,False,-0.013301,-0.013390,5.188340
1791057,2024-12-17,ZTS,170.789993,170.789993,177.320007,169.520004,175.000000,3018300.0,False,-0.028554,-0.028969,5.011640
1791058,2024-12-18,ZTS,167.009995,167.009995,170.940002,166.899994,170.070007,2728300.0,False,-0.022132,-0.022381,4.878588
1791059,2024-12-19,ZTS,163.520004,163.520004,167.440002,163.369995,166.589996,3713700.0,False,-0.020897,-0.021118,4.755744


Moving avgs and rolling volatility:

In [ ]:
stocks["sma_20"]=grouped.transform(lambda x: x.rolling(20).mean())

stocks["sma_50"]=grouped.transform(lambda x: x.rolling(50).mean())

#Volatility:
"""It will use the LOG Return. I want 21-day rolling volatility, representing a month (252/12), so that means the std dev of 21 actual
returns. Since each symbols' first log_return is NaN (nothing to compare it to), pandas .rolling(21) requires
21 non null values, not just 21 rows. That only happens when we are in row 22 (we have 21 values up to that
point)"""

stocks["rolling_vol_21"]=stocks.groupby("symbol")["log_return"].transform(lambda x: x.rolling(window=21, min_periods=21).std())

In [7]:
#to confirm the first values for sma_20 and vol_21 appear in the 20th and 22nd rows
stocks[stocks["symbol"]=="A"].iloc[18:26]

,date,symbol,adj_close,close,high,low,open,volume,no_trading,simple_return,log_return,cumulative_return,sma_20,sma_50,rolling_vol_21
18,2010-01-29,A,17.958029,20.050072,20.865522,19.971388,20.693848,5923046.0,False,-0.023005,-0.023273,-0.104473,NaN,NaN,NaN
19,2010-02-01,A,18.662767,20.836910,20.972818,20.121601,20.143063,5585429.0,False,0.039244,0.038493,-0.069329,19.320740,NaN,NaN
20,2010-02-02,A,18.899818,21.101574,21.165951,20.808298,20.901287,5884741.0,False,0.012702,0.012622,-0.057508,19.263079,NaN,NaN
21,2010-02-03,A,18.925446,21.130186,21.380545,21.001431,21.022890,6085634.0,False,0.001356,0.001355,-0.056230,19.217591,NaN,0.017006
22,2010-02-04,A,18.624323,20.793991,21.065807,20.701000,20.958511,6632392.0,False,-0.015911,-0.016039,-0.071246,19.160571,NaN,0.017165
23,2010-02-05,A,18.688398,20.865522,20.929899,20.157368,20.679543,9546662.0,False,0.003440,0.003435,-0.068051,19.108036,NaN,0.017222
24,2010-02-08,A,18.707617,20.886980,21.230330,20.836910,20.894135,6924154.0,False,0.001028,0.001028,-0.067093,19.056782,NaN,0.017238
25,2010-02-09,A,18.842157,21.037195,21.258942,20.872675,21.123034,5150651.0,False,0.007192,0.007166,-0.060384,19.011615,NaN,0.017364


Calendar features:

In [ ]:
stocks["day_of_week"]=stocks["date"].dt.dayofweek
#Monday = 0 in pandas. That's why -1 was used in sql, because in there Monday = 1

stocks["day_name"]=stocks["date"].dt.day_name()
stocks["month"]=stocks["date"].dt.month
stocks["quarter"]=stocks["date"].dt.quarter
stocks["year"]=stocks["date"].dt.year

stocks

,date,symbol,adj_close,close,high,low,open,volume,no_trading,simple_return,log_return,cumulative_return,sma_20,sma_50,rolling_vol_21,day_of_week,day_name,month,quarter,year
0,2010-01-04,A,20.053030,22.389128,22.625179,22.267525,22.453505,3815561.0,False,NaN,NaN,0.000000,NaN,NaN,NaN,0,Monday,1,1,2010
1,2010-01-05,A,19.835201,22.145924,22.331903,22.002861,22.324751,4186031.0,False,-0.010863,-0.010922,-0.010863,NaN,NaN,NaN,1,Tuesday,1,1,2010
2,2010-01-06,A,19.764729,22.067240,22.174536,22.002861,22.067240,3243779.0,False,-0.003553,-0.003559,-0.014377,NaN,NaN,NaN,2,Wednesday,1,1,2010
3,2010-01-07,A,19.739098,22.038628,22.045780,21.816881,22.017166,3095172.0,False,-0.001297,-0.001298,-0.015655,NaN,NaN,NaN,3,Thursday,1,1,2010
4,2010-01-08,A,19.732698,22.031473,22.067240,21.745352,21.917025,3733918.0,False,-0.000324,-0.000324,-0.015974,NaN,NaN,NaN,4,Friday,1,1,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1791056,2024-12-16,ZTS,175.809998,175.809998,179.289993,175.789993,177.990005,1938800.0,False,-0.013301,-0.013390,5.188340,176.644001,180.391746,0.008133,0,Monday,12,4,2024
1791057,2024-12-17,ZTS,170.789993,170.789993,177.320007,169.520004,175.000000,3018300.0,False,-0.028554,-0.028969,5.011640,176.362501,180.070801,0.010310,1,Tuesday,12,4,2024
1791058,2024-12-18,ZTS,167.009995,167.009995,170.940002,166.899994,170.070007,2728300.0,False,-0.022132,-0.022381,4.878588,175.935001,179.672460,0.011092,2,Wednesday,12,4,2024
1791059,2024-12-19,ZTS,163.520004,163.520004,167.440002,163.369995,166.589996,3713700.0,False,-0.020897,-0.021118,4.755744,175.327501,179.148451,0.011801,3,Thursday,12,4,2024


### INDEX table:

Index returns:

In [9]:
index["simple_return"]=index["sp500"].pct_change()
index["log_return"]=np.log(index["sp500"]/index["sp500"].shift(1))


index

,date,sp500,simple_return,log_return
0,2014-12-22,2078.54,NaN,NaN
1,2014-12-23,2082.17,0.001746,0.001745
2,2014-12-24,2081.88,-0.000139,-0.000139
3,2014-12-26,2088.77,0.003310,0.003304
4,2014-12-29,2090.57,0.000862,0.000861
...,...,...,...,...
2512,2024-12-16,6074.08,0.003799,0.003792
2513,2024-12-17,6050.61,-0.003864,-0.003871
2514,2024-12-18,5872.16,-0.029493,-0.029937
2515,2024-12-19,5867.08,-0.000865,-0.000865


Calendar

In [10]:
index["day_of_week"]=index["date"].dt.dayofweek
#Monday = 0 in pandas. That's why we used -1 in sql because in there Monday = 1

index["day_name"]=index["date"].dt.day_name()
index["month"]=index["date"].dt.month
index["quarter"]=index["date"].dt.quarter
index["year"]=index["date"].dt.year

index

,date,sp500,simple_return,log_return,day_of_week,day_name,month,quarter,year
0,2014-12-22,2078.54,NaN,NaN,0,Monday,12,4,2014
1,2014-12-23,2082.17,0.001746,0.001745,1,Tuesday,12,4,2014
2,2014-12-24,2081.88,-0.000139,-0.000139,2,Wednesday,12,4,2014
3,2014-12-26,2088.77,0.003310,0.003304,4,Friday,12,4,2014
4,2014-12-29,2090.57,0.000862,0.000861,0,Monday,12,4,2014
...,...,...,...,...,...,...,...,...,...
2512,2024-12-16,6074.08,0.003799,0.003792,0,Monday,12,4,2024
2513,2024-12-17,6050.61,-0.003864,-0.003871,1,Tuesday,12,4,2024
2514,2024-12-18,5872.16,-0.029493,-0.029937,2,Wednesday,12,4,2024
2515,2024-12-19,5867.08,-0.000865,-0.000865,3,Thursday,12,4,2024


### Sanity checks:

In [11]:
#Number of rows should be the same before and after transformations:

len(stocks)
# 1791061 - it is the same before these transformations


1791061

In [12]:
#First row of every symbol should have nulls for simple_return, log_return and 0 for cumulative_return:

stocks.groupby("symbol")[["symbol","simple_return", "log_return", "cumulative_return"]].head(2)




,symbol,simple_return,log_return,cumulative_return
0,A,NaN,NaN,0.000000
1,A,-0.010863,-0.010922,-0.010863
3768,AAPL,NaN,NaN,0.000000
3769,AAPL,0.001729,0.001728,0.001729
7536,ABBV,NaN,NaN,0.000000
...,...,...,...,...
1780533,ZBH,0.031656,0.031165,0.031656
1784300,ZBRA,NaN,NaN,0.000000
1784301,ZBRA,-0.001744,-0.001745,-0.001744
1788068,ZTS,NaN,NaN,0.000000


In [ ]:
#The above code only shows the first two rows. 
# If there are any nulls somewhere after those, it won't flag.
#To make sure there are no nulls after the first row of each symbol:

stocks.groupby("symbol")[["simple_return", "log_return", "cumulative_return"]
                         ].apply(lambda x: x.iloc[1:].isnull().any().any()).any()

#why the second .any():
    #without it i would get a df with each symbol as rows and the 3 return columns.
    #they would have true or false depending if that symbol has NaNs in that column.
    #with the second .any() we collapse to one simple flag, giving true or false if that
    # symbol has any NaN in any of those 3 columns.
    #The 3rd any() i added because it takes all results for all symbols and the consolidated
    # result from the second any() and just gives me one True or False (False = no NaN
    # in all of the results)

False

The number of nulls of each of the created columns:

In [ ]:
stocks.groupby("symbol")[["simple_return", "log_return", "cumulative_return", "sma_20", "sma_50", "rolling_vol_21"]].apply(lambda x: x.isnull().sum())


,simple_return,log_return,cumulative_return,sma_20,sma_50,rolling_vol_21
symbol,,,,,,
A,1,1,0,19,49,21
AAPL,1,1,0,19,49,21
ABBV,1,1,0,19,49,21
ABNB,1,1,0,19,49,21
ABT,1,1,0,19,49,21
...,...,...,...,...,...,...
XYL,1,1,0,19,49,21
YUM,1,1,0,19,49,21
ZBH,1,1,0,19,49,21


Stocks distribution:

In [ ]:
stocks["simple_return"].describe()


count    1.790559e+06
mean     7.097182e-04
std      1.973403e-02
min     -6.184778e-01
25%     -8.020942e-03
50%      7.556528e-04
75%      9.515196e-03
max      7.459324e-01
Name: simple_return, dtype: float64

### Saving the new tables:

In [ ]:
destination = Path(r"filepath\csv_features")

stocks.to_csv(destination / "stock_features.csv", index=False)
index.to_csv(destination / "index_features.csv", index=False)